<a href="https://colab.research.google.com/github/galipaydin/pes517/blob/main/notebooks/lab03_data_literacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 517 · Lab 3 — Clean it, describe it, plot it
**Artificial Intelligence Applications in Sport · Week 3**

⏱️ About 70 minutes · 🧪 This notebook creates its own data — nothing can fail to download

> **First:** `File → Save a copy in Drive`. Cells marked ✏️ **EDIT ME** are the ones you change.


### What you will produce
A season of athlete-monitoring data, described honestly, with one figure — and a written statement of which
data-quality problem in it would most distort an injury model.

**One rule for the whole course:** *look at the data before you model it.* Every professional disaster in sport
analytics starts with somebody who did not.


---
## Block 1 — Setup and data

One squad, 30 athletes, 26 weeks, 6 sessions a week — about 4,700 athlete-sessions.
The missingness in this file is deliberate and realistic; you will find it in Block 3.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

RNG = np.random.default_rng(517)
N_ATH, N_WEEKS, SESS = 30, 26, 6
POS  = ['GK','DF','MF','FW']

ath = pd.DataFrame({
    'athlete_id':  [f'A{i:02d}' for i in range(N_ATH)],
    'position':    RNG.choice(POS, N_ATH, p=[.10,.32,.35,.23]),
    'age':         RNG.integers(18, 33, N_ATH),
    'body_mass_kg':np.round(RNG.normal(76, 7, N_ATH), 1),
})
ath['frailty'] = RNG.normal(0, 1, N_ATH)          # real risk, never measured

rows = []
for _, a in ath.iterrows():
    for wk in range(1, N_WEEKS + 1):
        for s in range(SESS):
            match = (s == SESS - 1)
            dur   = RNG.normal(95 if match else 78, 12)
            rpe   = np.clip(RNG.normal(8.2 if match else 5.9, 1.3), 1, 10)
            dist  = np.clip(RNG.normal(9800 if match else 5400, 1100), 500, None)
            hsd   = np.clip(dist * RNG.normal(.09 if match else .05, .02), 0, None)
            rows.append(dict(
                date = pd.Timestamp('2026-08-03') + pd.Timedelta(weeks=wk-1, days=s),
                athlete_id = a['athlete_id'], position = a['position'],
                age = a['age'], body_mass_kg = a['body_mass_kg'],
                session_type = 'match' if match else 'training',
                duration_min = round(dur), rpe = round(rpe, 1),
                total_distance_m = round(dist), high_speed_distance_m = round(hsd),
                accelerations = int(np.clip(RNG.normal(46 if match else 28, 9), 0, None)),
                sleep_h = round(np.clip(RNG.normal(7.4, .9), 4, 10), 1),
                wellness_1_5 = int(np.clip(round(RNG.normal(3.6, .9)), 1, 5)),
                soreness_1_5 = int(np.clip(round(RNG.normal(2.4 + 1.0*match, .9)), 1, 5)),
                _frailty = a['frailty'], _match = int(match)))

df = pd.DataFrame(rows)
df['session_load'] = df.rpe * df.duration_min

# --- realistic missingness: wellness is NOT missing at random ---
prev_match = df.groupby('athlete_id')['_match'].shift(1).fillna(0).astype(bool)
p_missing  = np.where(prev_match, 0.23, 0.05)
df.loc[RNG.random(len(df)) < p_missing, 'wellness_1_5'] = np.nan
df.loc[RNG.random(len(df)) < 0.04, 'sleep_h'] = np.nan

df = df.drop(columns=['_frailty','_match'])
print(f'{len(df):,} athlete-sessions · {df.athlete_id.nunique()} athletes · '
      f'{df.date.min():%d %b %Y} to {df.date.max():%d %b %Y}')


---
## Block 2 — Look at it

Four commands. You will use these four at the start of every dataset for the rest of your career.


In [ ]:
print('--- shape ---');            print(df.shape)
print('\n--- columns and types ---'); print(df.dtypes)
print('\n--- numeric summary ---');   print(df.describe().round(1).T[['mean','std','min','max']])
df.head()


---
## Block 3 — Missingness is never random

Count what is missing. Then ask **when** it is missing, which is the question that matters.


In [ ]:
print('Missing values per column:')
print(df.isna().sum()[lambda s: s > 0].to_string())
print(f'\nOverall wellness missingness: {df.wellness_1_5.isna().mean():.1%}')

# is it missing at random? compare the day after a match with every other day
df = df.sort_values(['athlete_id','date'])
after_match = df.groupby('athlete_id')['session_type'].shift(1).eq('match')

print(f'  on days after a MATCH   : {df.loc[after_match,  "wellness_1_5"].isna().mean():.1%}')
print(f'  on every other day      : {df.loc[~after_match, "wellness_1_5"].isna().mean():.1%}')


### Why this matters more than it looks

Wellness is missing four to five times more often on the day after a match — precisely the days on which an athlete
is most fatigued and most at risk. The missing values are not noise; they carry information about the outcome you
want to predict.

Delete those rows and you delete the hardest days from your dataset. Fill them with the column mean and you tell
the model that post-match days were average. Both choices silently change your answer, and neither appears in the
methods section of most papers.


---
## Block 4 — Weekly aggregation

Session rows are not what a coach plans in. Weeks are.


In [ ]:
df['week'] = df.date.dt.isocalendar().week.astype(int)

weekly = (df.groupby(['athlete_id','week'])
            .agg(load          = ('session_load','sum'),
                 distance_km   = ('total_distance_m', lambda s: s.sum()/1000),
                 hsd_m         = ('high_speed_distance_m','sum'),
                 sessions      = ('session_load','size'),
                 mean_wellness = ('wellness_1_5','mean'))
            .reset_index())

print(weekly.head())
print(f'\n{len(weekly)} athlete-weeks')


---
## ✏️ EDIT ME — one athlete, one season


In [ ]:
ATHLETE = 'A07'   # ✏️ EDIT ME — any id from A00 to A29
WINDOW  = 4       # ✏️ EDIT ME — rolling mean length in weeks. Try 1, then 4, then 12.

w = weekly[weekly.athlete_id == ATHLETE].sort_values('week')
roll = w.load.rolling(WINDOW, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(w.week, w.load, color='#c9d6dd', label='weekly load (RPE × min)')
ax.plot(w.week, roll, color='#8A1C2B', lw=2.2, label=f'{WINDOW}-week rolling mean')
ax.set_xlabel('ISO week'); ax.set_ylabel('Session load (AU)')
ax.set_title(f'{ATHLETE} — season load profile')
ax.legend(frameon=False); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

print(f'{ATHLETE}: mean {w.load.mean():.0f} AU/week, peak {w.load.max():.0f} AU in week {int(w.loc[w.load.idxmax(),"week"])}')


---
## Block 5 — Break it 💥

Run the cell above three times, with `WINDOW` set to **1**, then **4**, then **12**.

Same athlete. Same season. Three different stories: a chaotic athlete, a periodised athlete, and an athlete who
barely varies. **The smoothing window is an analytical choice that changes the conclusion, and it is almost never
reported.** You will meet this again in Week 9, when we look at what the acute:chronic workload ratio really is.


---
## Block 6 — The squad at a glance


In [ ]:
squad = (weekly.groupby('athlete_id')
               .agg(mean_load=('load','mean'), peak_load=('load','max'),
                    weeks=('week','size'), wellness=('mean_wellness','mean'))
               .join(df.groupby('athlete_id')['position'].first()))

fig, ax = plt.subplots(figsize=(9, 4.2))
colors = {'GK':'#8A1C2B','DF':'#1B6470','MF':'#9A5714','FW':'#57504D'}
for pos, grp in squad.groupby('position'):
    ax.scatter(grp.mean_load, grp.peak_load, s=64, label=pos, color=colors[pos])
ax.set_xlabel('Mean weekly load (AU)'); ax.set_ylabel('Peak weekly load (AU)')
ax.set_title('Squad load profile by position'); ax.legend(frameon=False, title='Position')
ax.grid(alpha=.3); plt.tight_layout(); plt.show()


---
## Interpretation ✍️ **(graded)**

About 150 words:

1. **Which data-quality problem in this file would most distort an injury model, and exactly how?**
2. What would you do about it — and what does your fix cost you?
3. Name one thing in this dataset that is *not* measured but clearly matters for injury risk.

---

**Your answer:**

*(replace this line)*


---
## AI Use Log

Complete honestly — it is part of the mark. Write *none* if you used no AI tool.

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
### Submitting

1. `File → Save a copy in Drive` · 2. `Share → Anyone with the link → Viewer` · 3. Paste the link into ODTÜClass.
